# Amharic Named Entity Recognition Datasets

This document lists the three Amharic NER datasets along with their specific entity tagging conventions. **Note:** Each dataset uses a different format for entity tags. You will need to preprocess them to a common schema before combining or using them in a unified model.

---

## 1. Amharic Named Entity Recognition Dataset (Hugging Face)

**Description:**  
This dataset uses a sequence feature structure where the entity tags are defined using the following `ClassLabel` specification:

```python
Sequence(
	feature=ClassLabel(names=[
		'O', 
		'B-PER', 'I-PER', 
		'B-ORG', 'I-ORG', 
		'B-LOC', 'I-LOC', 
		'B-TIME', 'I-TIME', 
		'B-TTL', 'I-TTL'
	], id=None),
	length=-1,
	id=None
)
```

**Link:**  [Amharic NER Dataset](https://huggingface.co/datasets/rasyosef/amharic-named-entity-recognition)

---

## 2. Masakhane NER Amharic Train Data

**Description:**  
This dataset uses a similar structure as the Hugging Face dataset, except that for time-related tags it uses `B/I-DATA` instead of `B-TIME`/`I-TIME`.

**Link:**  [Masakhane NER Amharic Train Data](https://github.com/masakhane-io/masakhane-ner/blob/main/data/amh/train.txt)

---

## 3. ANEC: An Amharic Named Entity Corpus

**Description:**  
This corpus uses a more descriptive, log form of tag names. For instance, instead of tags like `B-ORG` or `I-PER`, it might use forms such as `ORGINIZATION` for organizations and `PERSON` for persons.

**Link:**  [ANEC: An Amharic Named Entity Corpus](https://github.com/Ebrahimc/ANEC-An-Amharic-Named-Entity-Corpus-/blob/main/Amharic%20NER%20Corpus.txt)

---

## Summary

Before merging these datasets, you must standardize their entity tagging schemes. For example, you may decide to:
- Convert all time tags to a common format (e.g., `B-TIME`/`I-TIME`).
- Normalize descriptive tag names (e.g., converting `B-ORGINIZATION`/`I-ORGINIZATION` to `B-ORG`/`I-ORG`).

In [ ]:
import re

def update_labels(input_filename, output_filename):
	label_map = {
		r'\bB-DATE\b': 'B-TIME',
		r'\bI-DATE\b': 'I-TIME',
		r'\bB-PERSON\b': 'B-PER',
		r'\bI-PERSON\b': 'I-PER',
		r'\bB-ORGANIZATION\b': 'B-ORG',
		r'\bI-ORGANIZATION\b': 'I-ORG',
		r'\bB-LOCATION\b': 'B-LOC',
		r'\bI-LOCATION\b': 'I-LOC'
	}

	with open(input_filename, 'r', encoding='utf-8') as infile:
		lines = infile.readlines()

	updated_lines = []
	
	for line in lines:
		for pattern, replacement in label_map.items():
			line = re.sub(pattern, replacement, line)
		updated_lines.append(line)

	with open(output_filename, 'w', encoding='utf-8') as outfile:
		outfile.writelines(updated_lines)

if __name__ == '__main__':
	input_file = 'input.txt'
	output_file = 'output.txt'
	update_labels(input_file, output_file)
	print(f"Updated file saved as {output_file}")

## Convert raw token-label text to CSV format for Amharic NER dataset

In [ ]:
import csv

def convert_to_csv(input_file, output_file):
	label2id = {
		'O': 0,
		'B-PER': 1,
		'I-PER': 2,
		'B-ORG': 3,
		'I-ORG': 4,
		'B-LOC': 5,
		'I-LOC': 6,
		'B-TIME': 7,
		'I-TIME': 8,
		'B-TTL': 9,
		'I-TTL': 10
	}
	rows = []
	sentence_id = 0
	with open(input_file, 'r', encoding='utf-8') as infile:
		for line in infile:
			line = line.strip()
			if not line:
				sentence_id += 1
				continue
			token_label = line.split()
			if len(token_label) != 2:
				continue
			token, label = token_label
			label_id = label2id.get(label, 0)
			rows.append([sentence_id, token, label_id])
	with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
		writer = csv.writer(csvfile)
		writer.writerow(["sentence_id", "token", "label_id"])
		writer.writerows(rows)

if __name__ == "__main__":
	convert_to_csv("output.txt", "output.csv")
	print("Conversion complete! Check 'output.csv'.")


## Generate Amharic NER Hugging Face dataset from CSV

In [ ]:
import csv
from datasets import Dataset, Features, Sequence, ClassLabel, Value, load_dataset

label2id = {
	'O': 0,
	'B-PER': 1,
	'I-PER': 2,
	'B-ORG': 3,
	'I-ORG': 4,
	'B-LOC': 5,
	'I-LOC': 6,
	'B-TIME': 7,
	'I-TIME': 8,
	'B-TTL': 9,
	'I-TTL': 10
}

all_labels = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-TIME", "I-TIME", "B-TTL", "I-TTL"]


def read_csv_and_group(csv_file):
	"""
	Read the CSV and group tokens/labels by sentence_id.
	Returns a dictionary of lists:
	  {
	    'tokens': list of lists of tokens,
	    'ner_tags': list of lists of label IDs
	  }
	"""
	sentences = {}
	with open(csv_file, 'r', encoding='utf-8') as f:
		reader = csv.DictReader(f)
		for row in reader:
			s_id = int(row['sentence_id'])
			token = row['token']
			label_id = int(row['label_id'])

			if s_id not in sentences:
				sentences[s_id] = {
					"tokens": [],
					"ner_tags": []
				}
			sentences[s_id]["tokens"].append(token)
			sentences[s_id]["ner_tags"].append(label_id)

	data = {
		"tokens": [],
		"ner_tags": []
	}
	for s_id in sorted(sentences.keys()):
		data["tokens"].append(sentences[s_id]["tokens"])
		data["ner_tags"].append(sentences[s_id]["ner_tags"])

	return data


def create_huggingface_dataset(data):
	"""
	Create a Hugging Face Dataset from our grouped data.
	We define 'tokens' as a list of strings,
	and 'ner_tags' as a list of class labels.
	"""
	features = Features({
		"tokens": Sequence(Value("string")),
		"ner_tags": Sequence(ClassLabel(names=all_labels))
	})

	dataset = Dataset.from_dict(data, features=features)
	return dataset


def main():
	data = read_csv_and_group("output.csv")

	dataset = create_huggingface_dataset(data)

	dataset.save_to_disk("amharic_ner_dataset")

if __name__ == "__main__":
	main()
